In [27]:
import pandas as pd
from sklearn.utils import resample
from sklearn.model_selection import  train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

In [28]:
df = pd.read_csv("../dataset/dataset_clean.csv", usecols=["title", 'body', 'priority'])

In [29]:
df["priority"].value_counts()

priority
low       95146
medium     9807
high       1956
Name: count, dtype: int64

In [30]:
df_low = df[df['priority'] == 'low']
df_med = df[df['priority'] == 'medium']
df_high = df[df['priority'] == 'high']

df_low_downsampled = resample(df_low, replace=False, n_samples=20000, random_state=42)

df_train = pd.concat([df_low_downsampled, df_med, df_high])

In [31]:
df_train["priority"].value_counts(normalize=True)

priority
low       0.629663
medium    0.308755
high      0.061581
Name: proportion, dtype: float64

In [32]:
df_train['text'] = df_train['title'].fillna('') + " " + df_train['body'].fillna('')

df_train['text'] = df_train['text'].str.strip().str.lower()

In [34]:
X_train, X_test, y_train, y_test = train_test_split(df_train['text'], df_train['priority'], test_size=0.2, random_state=42, stratify=df_train['priority'])

pipe = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=10000, stop_words='english')),
    ('svc', LinearSVC(class_weight='balanced', random_state=42))
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

        high       0.19      0.31      0.23       391
         low       0.75      0.74      0.74      4000
      medium       0.46      0.41      0.43      1962

    accuracy                           0.61      6353
   macro avg       0.46      0.49      0.47      6353
weighted avg       0.62      0.61      0.61      6353



In [38]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin

class UrgencyFeatures(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.high_indicators = ['crash', 'blocker', 'urgent', 'emergency', 'regression', 'vulnerability', 'panic', 'broken', 'production', 'down', 'security']

    def fit(self, X, y=None):
        return self

    def transform(self, X):  # changed Self to self
        # Apply .astype(int) on the numpy array, not the list comprehension
        features = np.array([any(word in str(text).lower() for word in self.high_indicators) for text in X], dtype=int)
        return features.reshape(-1, 1)


In [39]:
from sklearn.pipeline import FeatureUnion

# Combine TF-IDF and your custom Urgency extracted features
combined_features = FeatureUnion([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=10000, stop_words='english')),
    ('urgency', UrgencyFeatures())
])

# Create the full pipeline
pipe2 = Pipeline([
    ('features', combined_features),
    ('svc', LinearSVC(class_weight='balanced', random_state=42))
])

# Train and evaluate
pipe2.fit(X_train, y_train)

y_pred2 = pipe2.predict(X_test)

print("Classification Report with Urgency Features:")
print(classification_report(y_test, y_pred2))


Classification Report with Urgency Features:
              precision    recall  f1-score   support

        high       0.19      0.30      0.23       391
         low       0.75      0.74      0.74      4000
      medium       0.46      0.41      0.43      1962

    accuracy                           0.61      6353
   macro avg       0.46      0.48      0.47      6353
weighted avg       0.62      0.61      0.61      6353

